# Laborator 0 · Datasetul (Online Retail II)
**Dataset:** Online Retail II (UCI) · **Metodologie:** CRISP-DM (fazele 1–3, comune tuturor laboratoarelor)

Acest notebook descrie **o singură dată** datele folosite în toate cele șase laboratoare: ce conțin, ce probleme au, cum le curățăm
și cum separăm istoricul de viitor. Rezultatele se salvează în `artefacte/` și sunt afișate de pagina „Datasetul" din aplicație.

| Fază CRISP-DM | Secțiune |
|---|---|
| 1. Înțelegerea sarcinii | 0 |
| 2. Înțelegerea datelor | 2 |
| 3. Pregătirea datelor | 3 (curățare), 4 (separarea istoric / viitor) |

## 0. Înțelegerea sarcinii (Business understanding)
Datele sunt tranzacțiile unui retailer online britanic care vinde cadouri și articole decorative, mulți clienți fiind angrosiști.
Toate laboratoarele pornesc din aceleași date și din același tabel de clienți, ca rezultatele să fie comparabile.

**Întrebări la care răspunde acest notebook:**
1. Ce conține fiecare coloană și cât din ea lipsește?
2. Ce rânduri **nu sunt vânzări de produse** (servicii, anulări, ajustări) și cum le tratăm?
3. Cum arată vânzările în timp și pe țări?
4. Cum separăm **istoricul** (din care calculăm variabilele) de **viitor** (din care calculăm ținta), ca modelele să nu „vadă" viitorul?

## 1. Setup

In [ ]:
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from baza_date import ARTEFACTE, SEED, ZILE_FEREASTRA, construieste_clienti, curata, incarca_brut, salveaza_fig

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.3f}".format)

try:                                   # `display` există în Jupyter/VS Code; în terminal îl înlocuim cu print
    display
except NameError:
    def display(obj):
        print(obj)

## 2. Înțelegerea datelor (Data understanding)
### 2.1 Încărcare și dicționarul coloanelor
Identificatorii (`Invoice`, `StockCode`, `Customer ID`) se citesc ca **text**: sunt etichete, nu cantități (ex.: `85123A`, `C581484`).

In [ ]:
df = incarca_brut()
print(f"{len(df):,} rânduri × {df.shape[1]} coloane | perioada {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")

descrieri = {
    "Invoice": "Numărul facturii (6 cifre). Prefix „C” = anulare/retur, prefix „A” = ajustare contabilă.",
    "StockCode": "Codul produsului (de regulă 5 cifre + opțional o literă); apar și coduri de servicii (POST, M, BANK CHARGES…).",
    "Description": "Denumirea produsului (text liber; un cod poate avea descrieri ușor diferite).",
    "Quantity": "Bucăți pe linie. Negativ la anulări și ajustări de stoc.",
    "InvoiceDate": "Data și ora facturii.",
    "Price": "Prețul unitar în lire sterline (£).",
    "Customer ID": "Identificatorul clientului (lipsește la o parte din rânduri: clienți neînregistrați).",
    "Country": "Țara clientului.",
}
dictionar = pd.DataFrame({
    "tip": df.dtypes.astype(str),
    "lipsă (%)": df.isna().mean() * 100,
    "valori distincte": df.nunique(),
    "exemplu": [str(df[c].dropna().iloc[0]) for c in df.columns],
    "descriere": [descrieri[c] for c in df.columns],
})
dictionar.index.name = "coloană"
dictionar.to_csv(ARTEFACTE / "lab0_coloane.csv")
display(dictionar)

### 2.2 Probleme de calitate
Documentația UCI spune că `StockCode` are 5 cifre și că doar facturile „C” sunt anulări. În date apar și altele: coduri de servicii, facturi „A”, prețuri sau cantități ≤ 0, rânduri fără client, rânduri **identice**.
**Duplicatele exacte** nu le eliminăm: o linie repetată poate fi legitimă (același articol scanat de două ori în aceeași comandă) și nu avem o coloană care să le distingă. Măsurăm doar cât contează.

In [ ]:
cod_serviciu = df["StockCode"].str.fullmatch(r"POST|DOT|M|C2|D|S|BANK CHARGES|ADJUST2?|AMAZONFEE|CRUK|B|TEST\d+|gift_0001_\d+", case=False)
probleme = pd.DataFrame({"rânduri": {
    "fără ID de client": int(df["Customer ID"].isna().sum()),
    "fără descriere": int(df["Description"].isna().sum()),
    "coduri de servicii / taxe / vouchere": int(cod_serviciu.sum()),
    "facturi de anulare (prefix C)": int(df["Invoice"].str.startswith("C").sum()),
    "facturi de ajustare (prefix A)": int(df["Invoice"].str.startswith("A").sum()),
    "cantitate ≤ 0": int((df["Quantity"] <= 0).sum()),
    "preț ≤ 0": int((df["Price"] <= 0).sum()),
    "rânduri identice (duplicate exacte)": int(df.duplicated().sum()),
}})
probleme["% din total"] = probleme["rânduri"] / len(df) * 100
probleme.index.name = "problemă"
probleme.to_csv(ARTEFACTE / "lab0_probleme.csv")
display(probleme)

## 3. Pregătirea datelor (Data preparation)
### 3.1 Curățare în straturi
Rândurile sunt **mutate**, nu șterse, în patru tabele, fiecare cu rolul lui: servicii și ajustări administrative; anulări (retururi); ajustări de stoc (preț sau cantitate ≤ 0); **vânzări curate** (singurele folosite pentru cheltuiala clienților). Un rând nu poate fi în două straturi (verificat cu `assert` în `baza_date.curata`).

In [ ]:
tranzactii, retururi, administrative, ajustari_stoc, contabilitate = curata(df)
contabilitate.to_csv(ARTEFACTE / "curatare.csv", index=False)
display(contabilitate)

### 3.2 Vânzările în timp și pe țări
Venitul lunar din vânzările curate (retururile se scad separat, ca să vedem și cât se returnează). Sezonalitatea (vârf în toamnă, înainte de sărbători) explică de ce fereastra de 90 de zile din urmă nu e „tipică".

In [ ]:
luna = tranzactii["InvoiceDate"].dt.to_period("M").astype(str)
lunar = pd.DataFrame({
    "venit vânzări (£)": tranzactii.groupby(luna)["Valoare"].sum(),
    "retururi (£)": retururi.groupby(retururi["InvoiceDate"].dt.to_period("M").astype(str))["Valoare"].sum().abs(),
    "facturi": tranzactii.groupby(luna)["Invoice"].nunique(),
    "clienți identificați": tranzactii.dropna(subset=["Customer ID"]).groupby(luna)["Customer ID"].nunique(),
}).fillna(0)
lunar["venit net (£)"] = lunar["venit vânzări (£)"] - lunar["retururi (£)"]
lunar.index.name = "luna"
lunar.to_csv(ARTEFACTE / "lab0_lunar.csv")
display(lunar)

venit_tara = tranzactii.groupby("Country")["Valoare"].sum().sort_values(ascending=False)
top = venit_tara.head(10)
tari = pd.DataFrame({
    "venit (£)": top,
    "% din venit": top / venit_tara.sum() * 100,
    "clienți": tranzactii.dropna(subset=["Customer ID"]).groupby("Country")["Customer ID"].nunique().reindex(top.index),
    "facturi": tranzactii.groupby("Country")["Invoice"].nunique().reindex(top.index),
})
tari.loc["Alte țări"] = [venit_tara.iloc[10:].sum(), venit_tara.iloc[10:].sum() / venit_tara.sum() * 100, np.nan, np.nan]
tari.index.name = "țara"
tari.to_csv(ARTEFACTE / "lab0_tari.csv")
display(tari)

### 3.3 Distribuțiile variabilelor
Salvăm un eșantion de linii și variabilele clienților, ca aplicația să poată desena histograme pentru orice variabilă. Toate au **coadă lungă** (puțini clienți și puține comenzi foarte mari), de aceea `log1p` apare în toate laboratoarele.

In [ ]:
esantion = tranzactii[["Quantity", "Price", "Valoare"]].sample(min(20000, len(tranzactii)), random_state=SEED)
esantion.to_csv(ARTEFACTE / "lab0_esantion_linii.csv", index=False)

clienti, CUTOFF, istoric_v, viitor_v = construieste_clienti(tranzactii, retururi)
VAR_CL = ["recenta_zile", "vechime_zile", "frecventa", "produse_distincte", "vanzari_brute", "valoare_medie_factura", "rata_retur", "tinta_cheltuiala"]
clienti[VAR_CL].to_csv(ARTEFACTE / "lab0_clienti_variabile.csv", index=False)
forma = pd.DataFrame({"medie": clienti[VAR_CL].mean(), "mediană": clienti[VAR_CL].median(), "maxim": clienti[VAR_CL].max(),
                      "asimetrie": clienti[VAR_CL].skew()})
forma.index.name = "variabilă (clienți)"
forma.to_csv(ARTEFACTE / "lab0_forma_clienti.csv")
display(forma)

## 4. Separarea istoricului de viitor (evitarea scurgerii de informație)
**Problema:** dacă o variabilă ar conține informație din perioada pe care vrem s-o prezicem, modelul ar părea excelent în laborator și ar eșua în realitate (*data leakage*).
**Soluția:** alegem o **dată de tăiere** (ultima dată din date − 90 de zile). Variabilele (recență, frecvență, valoare medie, retururi…) se calculează **doar din istoric**, iar ținta (dacă a cumpărat, cât a cheltuit) **doar din ultimele 90 de zile**. Aceeași dată de tăiere se folosește în toate laboratoarele.

In [ ]:
sfarsit = tranzactii["InvoiceDate"].max()
inceput = tranzactii["InvoiceDate"].min()
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.barh([0], [(CUTOFF - inceput).days], left=[0], color="#4c78a8", height=0.5)
ax.barh([0], [(sfarsit.normalize() - CUTOFF).days], left=[(CUTOFF - inceput).days], color="#f58518", height=0.5)
ax.text((CUTOFF - inceput).days / 2, 0, f"ISTORIC → variabilele (x)\n{inceput.date()} … {CUTOFF.date()}", ha="center", va="center", color="w", fontsize=10)
ax.text((CUTOFF - inceput).days + (sfarsit.normalize() - CUTOFF).days / 2, 0, f"VIITOR → ținta (y)\nultimele {ZILE_FEREASTRA} zile", ha="center", va="center", color="w", fontsize=9)
ax.axvline((CUTOFF - inceput).days, color="k", ls="--"); ax.set_yticks([]); ax.set_xlabel("zile de la începutul datelor")
ax.set_title("Data de tăiere: ce se vede la antrenare și ce se prezice")
salveaza_fig("lab0_axa_timp")

n_cumparat = int(clienti["a_cumparat"].sum())
separare = pd.DataFrame({"valoare": {
    "data de tăiere": str(CUTOFF.date()), "ultima dată din date": str(sfarsit.date()),
    "clienți cu istoric": f"{len(clienti):,}", "dintre care cumpără în următoarele 90 de zile": f"{n_cumparat:,} ({n_cumparat / len(clienti):.1%})",
    "rânduri de vânzări în istoric": f"{len(istoric_v):,}", "rânduri de vânzări în viitor": f"{len(viitor_v):,}"}})
separare.index.name = "element"
separare.to_csv(ARTEFACTE / "lab0_separare.csv")
display(separare)

## 5. Salvarea rezumatului
Cifrele-cheie pe care le afișează aplicația (pagina „Datasetul" și pagina de start).

In [ ]:
n_ret_fara = int(retururi["Customer ID"].isna().sum())
info0 = {
    "randuri_brute": int(len(df)), "randuri_curate": int(len(tranzactii)), "facturi_curate": int(tranzactii["Invoice"].nunique()),
    "produse": int(tranzactii["StockCode"].nunique()), "clienti_id": int(df["Customer ID"].nunique()), "tari": int(df["Country"].nunique()),
    "data_min": str(inceput.date()), "data_max": str(sfarsit.date()), "data_taiere": str(CUTOFF.date()), "zile_fereastra": ZILE_FEREASTRA,
    "pct_fara_client": float(df["Customer ID"].isna().mean() * 100), "duplicate_exacte": int(df.duplicated().sum()),
    "pct_duplicate": float(df.duplicated().mean() * 100), "pct_retururi": float(len(retururi) / len(df) * 100),
    "pct_venit_uk": float(venit_tara.get("United Kingdom", 0) / venit_tara.sum() * 100),
    "venit_total": float(tranzactii["Valoare"].sum()), "clienti_cu_istoric": int(len(clienti)), "pct_cumpara_viitor": float(clienti["a_cumparat"].mean() * 100),
    "luna_maxima": str(lunar["venit vânzări (£)"].idxmax()), "pct_venit_luna_maxima": float(lunar["venit vânzări (£)"].max() / lunar["venit vânzări (£)"].sum() * 100),
    "retururi_fara_client": n_ret_fara,
}
(ARTEFACTE / "info_lab0.json").write_text(json.dumps(info0, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.Series(info0, name="valoare").to_frame())

## 6. Concluzii și limite
- Din rândurile brute, doar vânzările curate (stratul 4) se folosesc pentru cheltuiala clienților; celelalte straturi sunt păstrate pentru laboratorul de anomalii (Laboratorul 6).
- Rândurile fără ID de client nu pot intra în analiza pe client (Laboratoarele 1–5 la nivel de client), dar rămân în analiza pe factură și pe zi (Laboratorul 6).
- Duplicatele exacte sunt păstrate (nu le putem deosebi de linii legitime repetate); impactul lor e raportat în 2.2.
- **Limite:** o singură dată de tăiere (rezultatele pot depinde de ea); ~2 ani de date, deci o singură sărbătoare de iarnă completă în fiecare an; țara clientului este cea din factură, nu neapărat cea de reședință.